In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np 
import seaborn as sns 
import folium
from datetime import datetime, timedelta

import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# load excel dataset
raw_data = pd.read_excel(
    '/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/UNHCR_Flow_Data.xlsx',
    sheet_name='DATA'
)

print(raw_data.shape)
raw_data[:10]

(107341, 10)


,origin,OriginISO,OriginName,asylum,AsylumISO,AsylumName,AsylumRegion,Population types,Year,Count
0,ALG,DZA,Algeria,MTA,MLT,Malta,Europe,REF,1962,5
1,ANG,AGO,Angola,COD,COD,Dem. Rep. of the Congo,West and Central Africa,REF,1962,20000
2,ANG,AGO,Angola,NAM,NAM,Namibia,Eastern and Southern Africa,REF,1962,277
3,BDI,BDI,Burundi,NAM,NAM,Namibia,Eastern and Southern Africa,REF,1962,13
4,CHI,CHN,China,NEP,NPL,Nepal,Asia and the Pacific,REF,1962,5
5,COB,COG,"Congo, Republic of",NAM,NAM,Namibia,Eastern and Southern Africa,REF,1962,8
6,COD,COD,Dem. Rep. of the Congo,NAM,NAM,Namibia,Eastern and Southern Africa,REF,1962,13
7,IRQ,IRQ,Iraq,MTA,MLT,Malta,Europe,REF,1962,17
8,SOM,SOM,Somalia,NAM,NAM,Namibia,Eastern and Southern Africa,REF,1962,5
9,SUD,SDN,Sudan,MTA,MLT,Malta,Europe,REF,1962,10


In [3]:
# -------------------------------------------------------------------
#                      changes to the raw data 
# -------------------------------------------------------------------

#    lower casing
# ------------------
# header to lowercase
raw_data.columns = raw_data.columns.str.lower()
# data to lowercase
for col in raw_data.select_dtypes(include='object'):
    raw_data[col] = raw_data[col].str.lower()

# check for duplicates
# ---------------------
print("number of duplicates found:", raw_data.duplicated().sum()) # ----> 0 duplicates

# check for missing values
# ---------------------
print(raw_data.isnull().sum()) 
# show rows with missing values
# print(raw_data[raw_data.isnull().any(axis=1)])
# remove rows with missing values
raw_data = raw_data.dropna()


# Sanity check it worked:
# ---------------------
# print(raw_data.isnull().sum())
raw_data[:10]


number of duplicates found: 0
origin               0
originiso            0
originname          35
asylum               0
asylumiso            0
asylumname           0
asylumregion        37
population types     0
year                 0
count                0
dtype: int64


,origin,originiso,originname,asylum,asylumiso,asylumname,asylumregion,population types,year,count
0,alg,dza,algeria,mta,mlt,malta,europe,ref,1962,5
1,ang,ago,angola,cod,cod,dem. rep. of the congo,west and central africa,ref,1962,20000
2,ang,ago,angola,nam,nam,namibia,eastern and southern africa,ref,1962,277
3,bdi,bdi,burundi,nam,nam,namibia,eastern and southern africa,ref,1962,13
4,chi,chn,china,nep,npl,nepal,asia and the pacific,ref,1962,5
5,cob,cog,"congo, republic of",nam,nam,namibia,eastern and southern africa,ref,1962,8
6,cod,cod,dem. rep. of the congo,nam,nam,namibia,eastern and southern africa,ref,1962,13
7,irq,irq,iraq,mta,mlt,malta,europe,ref,1962,17
8,som,som,somalia,nam,nam,namibia,eastern and southern africa,ref,1962,5
9,sud,sdn,sudan,mta,mlt,malta,europe,ref,1962,10


In [ ]:
# ---------------------------------------------------------
#                 Deal with annoying names
# ---------------------------------------------------------

# change "originname"
name_edits = {
    "côte d'ivoire": "ivory coast",
    "iran (islamic rep. of)": "iran",
    "netherlands (kingdom of the)": "nederland",
    "state of palestine": "palestine",
    "lao people's dem. rep": "laos",
    "russian federation": "russia",
    "serbia and kosovo: s/res/1244 (1999)": "serbia/kosovo",
    "syrian arab rep.": "syria",
    "bolivia (plurinational state of)": "bolivia",
    "dem. rep. of the congo": "dr congo",
    "congo, republic of": "congo",
    "china, hong kong sar": "hong kong",
    "rep. of korea": "south korea",
    "dem. people's rep. of korea": "north korea",
    "central african rep.": "central african republic",
    "dominican rep.": "dominican republic",
    "united rep. of tanzania": "tanzania",
    "venezuela (bolivarian republic of)": "venezuela",
    "rep. of moldova": "moldova"
}

# count number of "serbia and kosovo: s/res/1244 (1999)" entries
print("number of 'serbia and kosovo: s/res/1244 (1999)' entries:", raw_data[raw_data['originname'] == "serbia and kosovo: s/res/1244 (1999)"].shape[0])
# count number of "côte d'ivoire" entries
print("number of 'côte d'ivoire' entries:", raw_data[raw_data['originname'] == "côte d'ivoire"].shape[0])
print()

# -----------------------------------
#            Apply changes
# -----------------------------------
# country name changes to the 'originname' column
raw_data['originname'] = raw_data['originname'].replace(name_edits)

# sanity check it worked
print("number of 'serbia/kosovo' entries after edit:", raw_data[raw_data['originname'] == "serbia/kosovo"].shape[0])
# number of "ivory coast" entries after edit
print("number of 'ivory coast' entries after edit:", raw_data[raw_data['originname'] == "ivory coast"].shape[0])


number of 'serbia and kosovo: s/res/1244 (1999)' entries: 1145
number of 'côte d'ivoire' entries: 1348

number of 'serbia/kosovo' entries after edit: 1145
number of 'ivory coast' entries after edit: 1348


In [5]:
# --------------------------------------------------------------
#             Export cleaned Data ----> CSV format
# --------------------------------------------------------------

clean_data = raw_data.copy()

# export cleaned data to csv
clean_data.to_csv('/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/cleaned_data.csv', index=False)

In [6]:
# -----------------------------------------------------------------
#                            filters 
# -----------------------------------------------------------------

#       Less than 100
# ----------------------------
# Count rows that have 'count' less than 100
less_than_100 = raw_data[raw_data['count'] < 100].shape[0]
print(f"Number of rows with 'count' less than 100: {less_than_100}")
# filter out rows where 'count' is less than 100
filtered_data = raw_data[raw_data['count'] >= 100].copy()


Number of rows with 'count' less than 100: 76864


In [7]:
# --------------------------------------------------------------
#            Export filtered Data ----> CSV format
# --------------------------------------------------------------

filtered_data = filtered_data.copy()
# export filtered data to csv
filtered_data.to_csv('/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/filtered_data.csv', index=False)